# Evaluating LLMs as Annotators

**Deep Learning in NLP**  
Groups 2–3 (RISS annotation scheme)  
23 June 2026

## General introduction

## Objectives
In this session, you will:

1. Load a human-annotated gold-standard dataset.
2. Formulate LLM inputs and generate annotations using an open-weight LLM.
3. Measure LLM–gold agreement.
4. Compare annotation quality across the following methods for an LLM:
   - zero-shot prompting;
   - few-shot prompting;
   - fine-tuning.

## Models

The notebook can be used with several open-weight models, including:

- `Qwen3-1.7B` <-- used as default
- `Qwen3.6-27B`
- `gemma-4-31b-it`
- `gpt-oss-120b`

## Expected Outcomes

By the end of the session, you should be able to:

- explain the difference between zero-shot, few-shot, and fine-tuned as LLM-annotation methods;
- inspect failure modes of LLM-based annotation methods across annotation categories;
- interpret evaluation metrics and compare methods performance.

---
**To keep the workflows manageable and self-contained, the session materials are split across separate notebooks for zero-shot, few-shot, and fine-tuning approaches.**

**Important:** Before making changes to this shared notebook, save your own copy:

`File → Save a copy in Drive`

**Check that you are running on GPU:**

`Runtime → Change runtime type → T4 GPU`

In [ ]:
# Download GitHub companion for this session to CoLab
from pathlib import Path
import os

REPO_DIR = Path("/content/DLDH-SocSci")

if not REPO_DIR.exists():
    !git clone https://github.com/kunilovskaya/DLDH-SocSci.git
else:
    print("Repository already present. Pulling latest changes...")
    %cd /content/DLDH-SocSci
    !git pull
    %cd /content
import sys
sys.path.insert(0, str(REPO_DIR))
# replace RUN with main_student_groups for the data from Groups2-3-on-riss-scheme (DL in NLP course) 15 June-30 June 2026
RUN = "trial_student_groups"

GOLD_DIR = REPO_DIR / "data" / RUN

PROMPT_DIR = REPO_DIR / "prompts"

print("=" * 80)
print("Repository loaded from:", REPO_DIR)
print(f"Working on =={RUN}==")
print("=" * 80)


Cloning into 'DLDH-SocSci'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 57 (delta 23), reused 45 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 59.36 KiB | 3.71 MiB/s, done.
Resolving deltas: 100% (23/23), done.
Repository loaded from: /content/DLDH-SocSci
Working on ==trial_student_groups==


In [ ]:
# load and explore gold dataset
import pandas as pd
from ast import literal_eval

df = pd.read_csv(GOLD_DIR / "tmp_gold_dataset.tsv", sep="\t")

df["group_tags"] = df["group_tags"].apply(lambda x: literal_eval(x) if pd.notna(x) else [])

df = df.sort_values("disagreement", ascending=True).reset_index(drop=True)
print(df[["sent_id", "disagreement", "group_tags", "text"]].head())
print(df.columns.tolist)

print("=" * 80)
print(f"Available data: {df.shape}")
print("=" * 80)

                  sent_id  disagreement  \
0  commons_2025_34523:019             0   
1    lords_2025_29433:055             0   
2    lords_1980_24467:016             0   
3    lords_1980_13663:074             0   
4  commons_2025_30142:004             1   

                                    group_tags  \
0                                           []   
1                                           []   
2                                           []   
3                                           []   
4  [AGE_RETIRED, DIS_DISABLED, INC_LOW_INCOME]   

                                                text  
0  The right hon Lady also asked about the Britis...  
1  We welcome decisions taken by certain Indian r...  
2  This world-wide business is of course very lar...  
3  To encourage encroachment of the private secto...  
4  I knew, for example, that 71% of pensioners wi...  
<bound method IndexOpsMixin.tolist of Index(['annotator', 'sent_id', 'updated_at', 'group_mentioned',
       '

## LLM Annotation pipeline
(note differences between human and LLM procedure)

MAIN TASK: use an instruction-tuned large language model (LLM) as an automatic annotator and compare its predictions against the human gold standard.

```text
groups
 ├─ group_mentioned
 ├─ multiple_groups
 └─ first_group
        │
        ▼
group_appealed?
        │
        ├── no → stop
        │
        └── yes
              ├─ opposed_groups
              ├─ pejorative
              ├─ group_tags
              └─ appeal classification
                   ├─ reasoning
                   ├─ polarity
                   └─ stance
```                   


## Part 1. Methods: Zero-shot vs. Few-shot prompting

Large language models can be adapted to annotation tasks in several ways.

### Zero-shot prompting

The model receives:

* a task description,
* label definitions,
* annotation instructions,
* the text to annotate.

No annotated examples are provided in the prompt.

The model must infer how to apply the annotation scheme solely from the instructions.

### Few-shot prompting

The model receives:

* the task description,
* label definitions,
* several annotated examples,
* the text to annotate.

The examples demonstrate how the annotation scheme should be applied and serve as in-context training data.

---

The prompts are loaded from the GitHub repository. Inspect the prompts before running.

The resulting annotations are compared to the human gold standard using the same function as for human-human setup.

Don't forget to save the results externally.


In [ ]:
# Load model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
import torch

print("=" * 80)
if torch.cuda.is_available():
    print("Are GPUs connected? -- True --")
    print(f"You are on {torch.cuda.get_device_name(0)}")
else:
    print("Are GPUs connected? -- False --")
    print("You are on CPU")
print("=" * 80)

Are GPUs connected? -- False --
You are on CPU


In [ ]:
# helper functions for input and output processing
# define generic inference function
def ask_llm(prompt, max_new_tokens=200):

    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

# define json parser
import json
import re

def parse_json(response):

    match = re.search(r"\{.*\}", response, re.DOTALL)

    if not match:
        return None

    return json.loads(match.group())

In [ ]:
# define prompts. Each prompt is tested independently at the moment
MENTIONS_PROMPT = """
List social groups mentioned in the TARGET sentence. Return noun phrases referring to groups of people.

{item}

Return JSON:

{{
    "groups": [""]
}}
"""

OPPOSED_PROMPT = """
TASK:
Decide if the DETECTED social groups in the TARGET sentence are opposed or not.
Assign yes when groups are explicitly contrasted, compared, or set against each other.

Examples:

- immigrant men versus native women -> yes
- unlike pensioners, students... -> yes
- refugees receive support whereas veterans do not -> yes

Otherwise assign no.

DETECTED social groups:
{groups}

RULES:
- Annotate only the TARGET sentence.
- Use PREV and NEXT only to understand of the TARGET sentence.
- Replace <your_answer> with either "yes" or "no".

{item}

Return JSON:

{{
    "opposed_groups": "<your_answer>"
}}
"""

PEJORATIVE_PROMPT = """
TASK:
Decide if the reference to the DETECTED group in the TARGET sentence is pejorative.

DEFINITION:
A pejorative reference is a derogatory, insulting, stigmatizing label or slur term.
Interpretation of regional labels depends on the context.

Examples:

- NAFRI instead of Nordafrikaner -> yes
- Welfare queen instead of welfare recipient -> yes
- Ossi for East Germans -> decide in context
- low-income families -> no

DETECTED group:
{group}

RULES:
- Annotate only the TARGET sentence.
- Use PREV and NEXT only to understand the TARGET sentence.
- Replace <your_answer> with either "yes" or "no".

{item}

Return JSON:

{{
    "pejorative": "<your_answer>"
}}
"""

TAXONOMY_PROMPT = """
TASK: Assign one or more of the following categories to DETECTED group in TARGET sentence:

* age
* gender_sexuality
* family
* disability
* ethnicity
* migration_citizenship
* religion
* geography
* education
* income
* occupation
* other

## RULES
- Annotate only the DETECTED group in the TARGET sentence.
- Use PREV and NEXT only to understand the TARGET sentence.
- Assign all applicable categories. A group may belong to multiple categories simultaneously.
- Use "other" only when none of the other categories apply.
- Return only category labels from the list above.

Examples:

* immigrant women → migration_citizenship, gender_sexuality
* disabled children → disability, age
* elderly refugees → age, migration_citizenship
* working mothers → family, occupation

DETECTED group:
{group}

{item}

Return JSON:

{{
    "group_tags": [""]
}}
"""

APPEAL_PROMPT = """
TASK:
Decide if the DETECTED group in the TARGET sentence is appealed to in a political argument.

DEFINITION:
A social-group appeal occurs when a social group is used as part of an argument rather than merely mentioned.

Assign yes when a social group is:
- affected by a problem,
- causing a problem,
- deserving praise, blame, protection, or support,
- benefiting from or harmed by a policy,
- the target of a proposed action.

Examples:
- "Immigrants are struggling." -> yes
- "Working mothers deserve support." -> yes
- "Immigrants live in Germany." -> no

DETECTED group:
{group}

RULES:
- Annotate only the DETECTED group in the TARGET sentence.
- Use PREV and NEXT only to understand the TARGET sentence.
- Replace <your_answer> with either "yes" or "no".

{item}

Return JSON:

{{
    "group_appealed": "<your_answer>"
}}
"""

CLASSIFY_APPEAL_PROMPT = """
TASK:
Decide how the DETECTED group is appealed in the TARGET sentence.

DETECTED group:
{group}

{item}

## Annotation categories

### Reasoning

What is the primary basis of the argument about the group?

* economic: resources, jobs, wages, skills, productivity, taxes, welfare, public spending, economic incentives, economic performance
* cultural: values, norms, identity, religion, language, traditions, belonging, social cohesion, national identity
* both: both economic and cultural reasoning are central to the argument
* unclear: insufficient information to determine the primary basis

Examples:

* "Immigration depresses wages." → economic
* "Immigration threatens our national identity." → cultural
* "Immigration depresses wages and undermines our way of life." → both

### Polarity

How is the group represented?

* pro-group: deserving support, protection, inclusion, resources, rights, recognition, or improved outcomes
* anti-group: deserving criticism, blame, exclusion, restrictions, reduced support, or worse outcomes
* mixed: both positive and negative evaluations are present and neither dominates

Examples:

* "We should support disadvantaged children." → pro-group
* "Benefits for the unemployed should be reduced." → anti-group
* "We should support unemployed people, but benefits should be more conditional." → mixed

### Stance

What is the speaker's position toward the group appeal?

* endorse: accepts and relies on the group-based reasoning
* reject: disputes or invalidates the group-based reasoning
* partial: partially accepts but limits, reframes, or contests the appeal
* unclear: speaker's stance cannot be determined

Examples:

* "Working families are struggling, so they deserve tax relief." → endorse
* "It is not immigrants causing housing shortages." → reject
* "Immigration may affect wages, but it is not the main driver." → partial

## Instructions

1. Consider only the first appealed social group.
2. Use the surrounding sentences only as context.
3. Annotate only the target sentence.
4. Select exactly one value for each category.
5. Return valid JSON only.

Return JSON:

{{
  "reasoning": "economic|cultural|both|unclear",
  "polarity": "pro-group|anti-group|mixed",
  "stance": "endorse|reject|partial|unclear"
}}
"""



In [ ]:
# get the sent_id with the lowest disagreement and detected appeal
# row = df[df["group_appealed"] == "yes"].iloc[3]
row = df[df["multiple_groups"] == "yes"].iloc[0]
sent_id = row["sent_id"]

print(row)

annotator                                                       gold
sent_id                                       commons_2025_43572:018
updated_at                          2026-06-21 13:14:56.267346+02:00
group_mentioned                                                  yes
group_appealed                                                   yes
intersectional                                                   yes
multiple_groups                                                  yes
opposed_groups                                                    no
pejorative                                                        no
group_tags                           [OCC_AGRICULTURE, OTH_NON_RISS]
reasoning                                         REASONING_ECONOMIC
polarity                                          POLARITY_PRO_GROUP
stance                                                STANCE_ENDORSE
text               As part of our commitment to strengthening vit...
left_context       The Government 

# === Experimenting with simpler prompts to find fomulations that work reliably ===

In [ ]:
SIMPLE_EXTRACT  = """
List social groups mentioned in the TARGET sentence. Return noun phrases referring to groups of people.
TARGET:
{text}

Return JSON:

{{
    "groups": [""]
}}
"""

prompt = SIMPLE_EXTRACT.format(text=row['text'])
# prompt = SIMPLE_EXTRACT.format(text="Disabled people will receive additional support.")
response = ask_llm(prompt)
data = parse_json(response)
print("\n=== MENTIONS PROMPT ===")
print(prompt)
print("=" * 30 + "\n")

print("\n=== MENTIONS RESULT ===")
print(data)
print("=" * 30 + "\n")


=== MENTIONS PROMPT ===

List social groups mentioned in the TARGET sentence. Return noun phrases referring to groups of people.
TARGET:
As part of our commitment to strengthening vital sectors across the agricultural and food industries, we have also announced measures to provide stability to farmers and workers in the UK’s poultry sector.

Return JSON:

{
    "groups": [""]
}



=== MENTIONS RESULT ===
{'groups': ['farmers', 'workers']}



In [ ]:
# define the expected outcome and implement the dicision hierarchy
from textwrap import fill

def run_prompt(prompt):
    response = ask_llm(prompt)
    data = parse_json(response)
    return data

def safe_run_prompt(prompt):

    try:
        return run_prompt(prompt)

    except Exception as e:
        print("FAILED")
        print(prompt)
        print(e)

        return None

def annotate_item(text, left_context="", right_context="",
                  result=None, verbose=True):

    if result is None:
        result = {}

    context = f"""
PREV:
{left_context}

TARGET:
{text}

NEXT:
{right_context}
"""
    # step 1. extract groups
    prompt = MENTIONS_PROMPT.format(item=text).strip()
    response = ask_llm(prompt)
    data = parse_json(response)
    if verbose:
        print("\n=== MENTIONS PROMPT ===")
        print(prompt)
        print("=" * 80 + "\n")

        print("\n=== MENTIONS RESULT ===")
        print(data)
        print("=" * 80 + "\n")

    groups = data["groups"]
    # store outcomes
    result["groups"] = groups

    # get the results for group_mentioned from the output
    result["group_mentioned"] = (
        "yes" if groups else "no"
    )

    # get the results for multiple_groups from the output
    result["multiple_groups"] = (
        "yes" if len(groups) > 1 else "no"
    )

    # early exit 1
    if not groups:
        if verbose:
            print("TARGET SENTENCE")
            print("-" * 80)
            print(fill(row["text"], width=80))
            print()
            print("No groups found. Early exit 1.")

        return result

    first_group = groups[0]

    # store the span for the group we are annotating
    result["first_group"] = first_group

    # The previous stage result are inserted into the next prompt.
    # step 2. decide if the DETECTED group is appealed to
    data = safe_run_prompt(APPEAL_PROMPT.format(item=context, group=first_group)).strip()
    appeal_detected = data["group_appealed"]
    result["group_appealed"] = appeal_detected

    # early exit 2
    if appeal_detected == "no":
        if verbose:
            print("TARGET SENTENCE")
            print("-" * 80)
            print(fill(row["text"], width=80))
            print()
            print("No appealed groups found. Early exit 2.")

        return result

    # step 3. check if the appealed group is opposed to other groups
    data = safe_run_prompt(OPPOSED_PROMPT.format(item=context, groups=groups)).strip()
    result["opposed_groups"] = data["opposed_groups"]

    # step 4. pejorative reference to first group which is detected as appealed?
    data = safe_run_prompt(PEJORATIVE_PROMPT.format(item=context, group=first_group)).strip()
    result["pejorative"] = data["pejorative"]

    # step 5. classify the first group: multiclass for each category
    data = safe_run_prompt(TAXONOMY_PROMPT.format(item=context, group=first_group)).strip()
    result["group_tags"] = data["group_tags"]
    if len(data["group_tags"]) > 2:
        result['intersectional'] = "yes"

    # step 6. run three multiclass classifications
    data = safe_run_prompt(CLASSIFY_APPEAL_PROMPT.format(item=context, group=first_group)).strip()
    result["reasoning"] = data["reasoning"]
    result["polarity"] = data["polarity"]
    result["stance"] = data["stance"]

    return result

In [ ]:
# run on one example
import time
start = time.time()

annotations = {
    "groups": [],
    "group_mentioned": "no",
    "multiple_groups": "no",
    "first_group": None,
    "group_appealed": "no",
    "opposed_groups": "no",
    "pejorative": "no",
    "group_tags": [],
    "intersectional": 'no',
    "reasoning": None,
    "polarity": None,
    "stance": None,
}

ann = annotate_item(row["text"], row["left_context"], row["right_context"],
                    result=annotations, verbose=True)
print(ann)

item_preds_df = pd.DataFrame([ann])

# post-process appeal tags
LABEL_MAPPING = {
    "reasoning": {
        "economic": "REASONING_ECONOMIC",
        "cultural": "REASONING_CULTURAL",
        "security": "REASONING_SECURITY",
        "unclear": "REASONING_UNCLEAR",
    },
    "polarity": {
        "pro-group": "POLARITY_PRO_GROUP",
        "anti-group": "POLARITY_ANTI_GROUP",
        "mixed": "POLARITY_MIXED",
        "unclear": "POLARITY_UNCLEAR"
    },
    "stance": {
        "endorse": "STANCE_ENDORSE",
        "oppose": "STANCE_OPPOSE",
        "unclear": "STANCE_UNCLEAR",
    }
}

# attach identifiers
item_preds_df.insert(0, "sent_id", sent_id)
# insert LLM's name as annotator
llm_annotator = model_name.split('/')[-1].lower()
item_preds_df.insert(0, "annotator", llm_annotator)

item_preds_df = item_preds_df.replace(LABEL_MAPPING)

print("\n" + "=" * 80)
print(item_preds_df)
print("=" * 80 + "\n")

end = time.time()
elapsed_time = (end - start) / 60
if elapsed_time < 60:
    print(f"\nTotal time: {elapsed_time:.1f} min")
else:
    print(f"\nTotal time: {int(elapsed_time // 60)} h {int(elapsed_time % 60)} min")


=== MENTIONS PROMPT ===

List social groups mentioned in the TARGET sentence. Return noun phrases referring to groups of people.

As part of our commitment to strengthening vital sectors across the agricultural and food industries, we have also announced measures to provide stability to farmers and workers in the UK’s poultry sector.

Return JSON:

{
    "groups": [""]
}



=== MENTIONS RESULT ===
{'groups': ['farmers', 'workers']}

{'groups': ['farmers', 'workers'], 'group_mentioned': 'yes', 'multiple_groups': 'yes', 'first_group': 'farmers', 'group_appealed': 'yes', 'opposed_groups': 'no', 'pejorative': 'no', 'group_tags': ['occupation'], 'intersectional': 'no', 'reasoning': 'economic', 'polarity': 'pro-group', 'stance': 'endorse'}

               groups group_mentioned multiple_groups first_group  \
0  [farmers, workers]             yes             yes     farmers   

  group_appealed opposed_groups pejorative    group_tags intersectional  \
0            yes             no         

In [ ]:
# compare to gold with the naked eye
from textwrap import fill

print("TARGET SENTENCE")
print("-" * 80)
print(fill(row["text"], width=80))

# post-process appeal tags
LABEL_MAPPING = {
    "reasoning": {
        "economic": "REASONING_ECONOMIC",
        "cultural": "REASONING_CULTURAL",
        "security": "REASONING_SECURITY",
        "unclear": "REASONING_UNCLEAR",
    },
    "polarity": {
        "pro-group": "POLARITY_PRO_GROUP",
        "anti-group": "POLARITY_ANTI_GROUP",
        "mixed": "POLARITY_MIXED",
        "unclear": "POLARITY_UNCLEAR"
    },
    "stance": {
        "endorse": "STANCE_ENDORSE",
        "oppose": "STANCE_OPPOSE",
        "unclear": "STANCE_UNCLEAR",
    }
}

item_preds_df = item_preds_df.replace(LABEL_MAPPING)

pred_row = item_preds_df.iloc[0]

print()
print(f'Detected group spans: {pred_row["groups"]}')
print()

print()
print(f'Group tags assigned to {pred_row["first_group"]}: {pred_row["group_tags"]}')
print()

print(f"{'Variable':<20} {'Pred':<10} {'Gold':<10} {'✓'}")
print("-" * 46)


for col in ["group_mentioned", "group_appealed", "multiple_groups",
            "intersectional", "opposed_groups", "pejorative",
            "reasoning", "polarity", "stance"]:
    ok = row[col] == pred_row[col]
    try:
      print(
          f"{col:<20} "
          f"{str(pred_row[col]):<10} "
          f"{str(row[col]):<10} "
          f"{'✓' if ok else '✗'}"
      )
    except ValueError:
      print(col)
      continue

TARGET SENTENCE
--------------------------------------------------------------------------------
As part of our commitment to strengthening vital sectors across the agricultural
and food industries, we have also announced measures to provide stability to
farmers and workers in the UK’s poultry sector.

Detected group spans: ['farmers', 'workers']


Group tags assigned to farmers: ['occupation']

Variable             Pred       Gold       ✓
----------------------------------------------
group_mentioned      yes        yes        ✓
group_appealed       yes        yes        ✓
multiple_groups      yes        yes        ✓
intersectional       no         yes        ✗
opposed_groups       no         no         ✓
pejorative           no         no         ✓
reasoning            REASONING_ECONOMIC REASONING_ECONOMIC ✓
polarity             POLARITY_PRO_GROUP POLARITY_PRO_GROUP ✓
stance               STANCE_ENDORSE STANCE_ENDORSE ✓


In [ ]:
# anything stored under `/content/` is temporary and disappears when the Colab session ends.
# downloading predictions for one item (one-row table)
from pathlib import Path
from google.colab import files

OUT_DIR = Path(REPO_DIR / "results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

item_preds_df.to_csv(OUT_DIR / f"preds_{sent_id}.tsv", sep="\t", index=False)


files.download(OUT_DIR / f"preds_{sent_id}.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# == Scaling to 52 examples ==

In [ ]:
def postprocess(llm_preds):
    llm_preds = llm_preds.copy()
    # post-process appeal tags
    LABEL_MAPPING = {
        "reasoning": {
            "economic": "REASONING_ECONOMIC",
            "cultural": "REASONING_CULTURAL",
            "security": "REASONING_SECURITY",
            "unclear": "REASONING_UNCLEAR",
        },
        "polarity": {
            "pro-group": "POLARITY_PRO_GROUP",
            "anti-group": "POLARITY_ANTI_GROUP",
            "mixed": "POLARITY_MIXED",
            "unclear": "POLARITY_UNCLEAR"
        },
        "stance": {
            "endorse": "STANCE_ENDORSE",
            "oppose": "STANCE_OPPOSE",
            "unclear": "STANCE_UNCLEAR",
        }
    }

    # attach identifiers
    llm_preds.insert(0, "sent_id", sent_id)
    # insert LLM's name as annotator
    llm_annotator = model_name.split('/')[-1].lower()
    llm_preds.insert(0, "annotator", llm_annotator)

    llm_preds = llm_preds.replace(LABEL_MAPPING)

    return llm_preds

In [ ]:
# run on all items
start = time.time()
results = []

for _, row in df.iterrows():

    ann = annotate_item(
        row["text"],
        row["left_context"],
        row["right_context"]
    )

    results.append(ann)
sample_preds_df = pd.DataFrame(results)

sample_preds_df = postprocess(sample_preds_df)

end = time.time()
elapsed_time = (end - start) / 60
if elapsed_time < 60:
    print(f"\nTotal time: {elapsed_time:.1f} min")
else:
    print(f"\nTotal time: {int(elapsed_time // 60)} h {int(elapsed_time % 60)} min")


=== MENTIONS PROMPT ===

List social groups mentioned in the TARGET sentence. Return noun phrases referring to groups of people.

The right hon Lady also asked about the British consular assistance.

Return JSON:

{
    "groups": [""]
}



=== MENTIONS RESULT ===
{'groups': []}

TARGET SENTENCE
--------------------------------------------------------------------------------
The right hon Lady also asked about the British consular assistance.

No groups found. Early exit 1.


In [ ]:
# re-use the same function that was used for triple human evaluation
import sys
sys.path.append("/content/DLDH-SocSci")
from iaa_and_gold import iaa_calculation

print(sys.path[-1])

binary_fields = [
    "group_mentioned",
    "group_appealed",
    "multiple_groups",
    "intersectional",
    "opposed_groups",
    "pejorative",
]

multiclass_fields = [
    "reasoning",
    "polarity",
    "stance",
]

gold_llm = pd.concat([df, sample_preds_df], axis=0)
iaa_res, _, _ = iaa_calculation(gold_llm, binary=binary_fields,
                                multilabel=[], multiclass=multiclass_fields,
                                min_freq=5, meth='library')

print(iaa_res)

In [ ]:
# anything stored under `/content/` is temporary and disappears when the Colab session ends.
from pathlib import Path
from google.colab import files

OUT_DIR = Path(REPO_DIR / "results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

sample_preds_df.to_csv(OUT_DIR / "sample_preds_df.tsv", sep="\t", index=False)
iaa_res.to_csv(OUT_DIR / f"gold_{llm_annotator}_iaa_res.tsv", sep="\t", index=False)

files.download(OUT_DIR / "sample_preds_df.tsv")

/content/DLDH-SocSci/results/one_binary_pred.tsv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sys
sys.exit("Stopping here")

# == End of zero-shot experimental branch ==
Feel free to copy the notebook and improve the results